Works but only sends one command

Reconnecting each time means it works. E.g. for each command: 

g = GilsonSession("192.168.1.101")
g.send_command("Get XY Position")
g.close()

In [1]:
import socket
import time
import xml.etree.ElementTree as ET

class GilsonSession:
    """
    A class to manage a session with a Gilson Ethernet-controlled device.
    Handles connection, command formatting, sending, and response parsing.
    """

    def __init__(self, ip, admin_port=50185):
        """
        Initialize the session and connect to the Gilson controller.

        :param ip: IP address of the Gilson device.
        :param admin_port: Port to initiate the session (default is 50185).
        """
        self.ip = ip
        self.admin_port = admin_port
        self.session_port = None
        self.session_socket = None
        self.sequence_number = 0
        self._connect()

    def _connect(self):
        """
        Establish the initial admin connection and obtain a session port.
        Then open a persistent session socket.
        """
        message = """
        <Message MessageId="1984" BlockCount="1" BlockId="0">
        <Gilson>
            <GilsonConnect>Gilson Ethernet Utility</GilsonConnect>
        </Gilson>
        </Message>
        """.replace('\n', '\r\n').strip()

        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(5)
            s.connect((self.ip, self.admin_port))
            s.sendall(message.encode('ascii'))
            response = s.recv(4096).decode('ascii')

        if "<ParameterName>ResponseValue</ParameterName>" in response:
            port = int(response.split("<ParameterValue>")[2].split("</ParameterValue>")[0])
            self.session_port = port

            # Open persistent session socket
            self.session_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self.session_socket.settimeout(5)
            self.session_socket.connect((self.ip, self.session_port))
        else:
            raise RuntimeError("Failed to retrieve assigned session port from Gilson.")

    def send_raw_command(self, xml_payload):
        """
        Send raw XML to the device and read the response.

        :param xml_payload: XML-formatted string command.
        :return: Full XML response string.
        """
        try:
            self.session_socket.sendall(xml_payload.encode('ascii'))
            time.sleep(0.5)  # small delay to allow processing
            buffer = b""
            while True:
                try:
                    part = self.session_socket.recv(8192)
                    if not part:
                        break
                    buffer += part
                    if b"</Message>" in part:
                        break
                except socket.timeout:
                    break
            response = buffer.decode('ascii', errors='ignore')
            print(f"Full XML response:\n{response}\n")
            return response
        except Exception as e:
            print(f"Error sending raw command:", e)
            return None

    def _get_next_sequence_number(self):
        """
        Increment and return the next sequence number.
        """
        self.sequence_number += 1
        return self.sequence_number

    def make_command(self, command_name, device="GX-27x", device_id=35, sequence_number=None, command_type="Local"):
        """
        Format a full XML command.

        :param command_name: Name of the command (e.g., 'Identify').
        :param device: Name of the device (default 'GX-27x').
        :param device_id: Device ID (default 35).
        :param sequence_number: Sequence number (auto-incremented if None).
        :param command_type: 'Local' or 'Immediate'.
        :return: XML string.
        """
        if sequence_number is None:
            sequence_number = self._get_next_sequence_number()

        message_id = int(time.time()) % 10000
        xml = f"""
        <Message MessageId="{message_id}" BlockCount="2" BlockId="0">
        <Gilson>
            <CommandData>
                <Commands>
                    <Command>
                        <CommandType>{command_type}</CommandType>
                        <CommandInfo>
                            <InstrumentInfo>
                                <DeviceName>{device}</DeviceName>
                                <DeviceId>{device_id}</DeviceId>
                            </InstrumentInfo>
                            <SequenceNumber>{sequence_number}</SequenceNumber>
                            <Synchronize>True</Synchronize>
                            <Selected>False</Selected>
                            <CommandXML>
                                <CommandName>{command_name}</CommandName>
                            </CommandXML>
                        </CommandInfo>
                    </Command>
                </Commands>
            </CommandData>
        </Gilson>
        </Message>
        <Message MessageId="{message_id}" BlockCount="2" BlockId="1">
        </Message>
        """.replace('\n', '\r\n').strip()
        return xml

    def send_command(self, command_name, device="GX-27x", device_id=35, sequence_number=None):
        """
        Send a 'Local' command to the device.
        """
        xml = self.make_command(command_name, device, device_id, sequence_number, command_type="Local")
        return self.send_raw_command(xml)

    def send_immediate_command(self, command_name, device="GX-27x", device_id=35, sequence_number=None):
        """
        Send an 'Immediate' command to the device.
        """
        xml = self.make_command(command_name, device, device_id, sequence_number, command_type="Immediate")
        return self.send_raw_command(xml)

    def close(self):
        """
        Close the session socket.
        """
        if self.session_socket:
            self.session_socket.close()
            self.session_socket = None


In [2]:
g = GilsonSession("192.168.1.101")

In [3]:
g.send_command("Admin", device="System", device_id=-1)

Full XML response:
<Message MessageId="18160" BlockCount="2" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>255.255.255.0</SubnetMask><DefaultGateway>192.168.1.1</DefaultGateway><MAC>00:40:9d:a2:56:92</MAC><LeaseInformation><LeaseActive>False</LeaseActive></LeaseInformation></SystemInformation><Devices><Device><DeviceName>GX-27x</DeviceName><DeviceId>35</DeviceId><SerialNumber>261J7N253</SerialNumber><Connected>True</Connected><Commands><Command><CommandName>Get Input Contacts</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Contact A</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Contact B</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Sa

'<Message MessageId="18160" BlockCount="2" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>255.255.255.0</SubnetMask><DefaultGateway>192.168.1.1</DefaultGateway><MAC>00:40:9d:a2:56:92</MAC><LeaseInformation><LeaseActive>False</LeaseActive></LeaseInformation></SystemInformation><Devices><Device><DeviceName>GX-27x</DeviceName><DeviceId>35</DeviceId><SerialNumber>261J7N253</SerialNumber><Connected>True</Connected><Commands><Command><CommandName>Get Input Contacts</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Contact A</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Contact B</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Safety</ParameterNam

In [8]:
g.send_immediate_command("Identify")

Full XML response:
<Message MessageId="18166" BlockCount="1" BlockId="0">
        </Message>
        <Message MessageId="2576" BlockCount="2" BlockId="1">
        </Message><Message MessageId="2582" BlockCount="2" BlockId="0">
        <Gilson>
            <CommandData>
                <Commands>
                    <Command>
                        <CommandType>Immediate</CommandType>
                        <CommandInfo>
                            <InstrumentInfo>
                                <DeviceName>GX-27x</DeviceName>
                                <DeviceId>35</DeviceId>
                            </InstrumentInfo>
                            <SequenceNumber>5</SequenceNumber>
                            <Synchronize>True</Synchronize>
                            <Selected>False</Selected>
                            <CommandXML>
                                <CommandName>Identify</CommandName>
                            </CommandXML>
                        </CommandI

'<Message MessageId="18166" BlockCount="1" BlockId="0">\r\n        </Message>\r\n        <Message MessageId="2576" BlockCount="2" BlockId="1">\r\n        </Message><Message MessageId="2582" BlockCount="2" BlockId="0">\r\n        <Gilson>\r\n            <CommandData>\r\n                <Commands>\r\n                    <Command>\r\n                        <CommandType>Immediate</CommandType>\r\n                        <CommandInfo>\r\n                            <InstrumentInfo>\r\n                                <DeviceName>GX-27x</DeviceName>\r\n                                <DeviceId>35</DeviceId>\r\n                            </InstrumentInfo>\r\n                            <SequenceNumber>5</SequenceNumber>\r\n                            <Synchronize>True</Synchronize>\r\n                            <Selected>False</Selected>\r\n                            <CommandXML>\r\n                                <CommandName>Identify</CommandName>\r\n                            </Command

In [5]:
g.send_command("Get XY Position")

Full XML response:
<Message MessageId="18164" BlockCount="1" BlockId="0">
        </Message>
        <Message MessageId="2552" BlockCount="2" BlockId="1">
        </Message><Message MessageId="2552" BlockCount="2" BlockId="0">
        <Gilson>
            <CommandData>
                <Commands>
                    <Command>
                        <CommandType>Local</CommandType>
                        <CommandInfo>
                            <InstrumentInfo>
                                <DeviceName>GX-27x</DeviceName>
                                <DeviceId>35</DeviceId>
                            </InstrumentInfo>
                            <SequenceNumber>3</SequenceNumber>
                            <Synchronize>True</Synchronize>
                            <Selected>False</Selected>
                            <CommandXML>
                                <CommandName>Get XY Position</CommandName>
                            </CommandXML>
                        </Comma

'<Message MessageId="18164" BlockCount="1" BlockId="0">\r\n        </Message>\r\n        <Message MessageId="2552" BlockCount="2" BlockId="1">\r\n        </Message><Message MessageId="2552" BlockCount="2" BlockId="0">\r\n        <Gilson>\r\n            <CommandData>\r\n                <Commands>\r\n                    <Command>\r\n                        <CommandType>Local</CommandType>\r\n                        <CommandInfo>\r\n                            <InstrumentInfo>\r\n                                <DeviceName>GX-27x</DeviceName>\r\n                                <DeviceId>35</DeviceId>\r\n                            </InstrumentInfo>\r\n                            <SequenceNumber>3</SequenceNumber>\r\n                            <Synchronize>True</Synchronize>\r\n                            <Selected>False</Selected>\r\n                            <CommandXML>\r\n                                <CommandName>Get XY Position</CommandName>\r\n                            </Comm

In [6]:
#g.close()